In [ ]:
!pip install torchmetrics -q

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torch import optim
from torch.optim.lr_scheduler import StepLR
from torchvision import transforms
import torchmetrics
import numpy as np
import cv2
from google.colab.patches import cv2_imshow
import matplotlib.pyplot as plt

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
!git clone https://github.com/facebookresearch/dinov3.git
!cp dinov3/hubconf.py .

In [ ]:
import sys
sys.path.append("dinov3")

In [ ]:
!wget http://www.agentspace.org/download/mydinov3.zip
!unzip -P dino mydinov3.zip

In [ ]:
backbone = torch.hub.load('.', 'dinov3_vits16', source='local', weights='dinov3_vits16_pretrain_lvd1689m.pth') # dinov3_vits16
backbone.to(device)
backbone.eval()

In [ ]:
def make_transform(resize_size = 768):
    to_tensor = transforms.ToTensor()
    resize = transforms.Resize((resize_size, resize_size), antialias=True)
    normalize = transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    )
    return transforms.Compose([to_tensor, resize, normalize])

img_size = 768
transform = make_transform(img_size)

In [ ]:
!pip install fastncut

In [ ]:
from fastncut import ncut, Ncut, targetFromMask

In [ ]:
# download dataset https://github.com/JizhiziLi/AIM
# Li, Jizhizi, Jing Zhang, and Dacheng Tao. "Deep Automatic Natural Image Matting." IJCAI. 2021.
# refer to the paper (https://arxiv.org/abs/2107.07235), the GitHub repo (https://github.com/JizhiziLi/AIM) or contacts Jizhizi Li at jili8515@uni.sydney.edu.au
!wget http://www.agentspace.org/download/AIM-500-20260316T160703Z-1-001.zip
!unzip AIM-500-20260316T160703Z-1-001.zip > /dev/null

In [ ]:
indices = []
samples = ["o_1ea2b894","o_35d8df37","o_542332e9","o_2b9227eb","o_2309157d","o_33d1515f","o_260f47c8","o_494867da","o_60104376","o_6449cd22"]

In [ ]:
# find all images
filenames = []
images = []
for root, dirs, files in os.walk("AIM-500/original"):
    for file in files:
        if file.endswith(".jpg"):
            filenames.append(file[:-4])
            image = cv2.imread(os.path.join("AIM-500/original",file))
            if file[:-4] in samples:
                indices.append(len(images))
            images.append(image)

In [ ]:
len(filenames), len(images)

In [ ]:
indices

In [ ]:
# find all sample inputs
inputs = []
for image in images:
    with torch.no_grad():
        blob = transform(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)).unsqueeze(0).to(device)
        feats = backbone.get_intermediate_layers(blob,norm=False)[0]
        feats = feats.view(feats.shape[0],blob.shape[2]//backbone.patch_size,blob.shape[3]//backbone.patch_size,feats.shape[-1])
        feats = feats.permute(0,3,1,2)
    inputs.append(feats[0].cpu())
    if (len(inputs) % 50 == 0):
        print(len(inputs))

In [ ]:
len(inputs)

In [ ]:
# unload model
backbone.cpu()
torch.cuda.empty_cache()

In [ ]:
# find all sample outputs
outputs = []
for root, dirs, files in os.walk("AIM-500/original"):
    for file in files:
        if file.endswith(".jpg"):
            mask = cv2.imread(os.path.join("AIM-500/mask",file[:-4]+'.png'),cv2.IMREAD_GRAYSCALE)
            threshold, mask = cv2.threshold(mask,0,255,cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            outputs.append(mask)

In [ ]:
len(outputs), outputs[0].shape

In [ ]:
def cv2_imshow_mask(image, mask, display=True):
    overlay = image.copy()
    overlay[mask > 0] = (0,255,255)
    result = cv2.addWeighted(overlay, 0.4, image, 0.6, 0)
    if display:
        cv2_imshow(result)
    else:
        return result

In [ ]:
# display inputs
fig, axs = plt.subplots(1, len(indices), figsize=(20, 10))
for i, index in enumerate(indices):
    axs[i].imshow(cv2.cvtColor(images[index],cv2.COLOR_BGR2RGB))
    axs[i].axis('off')
    axs[i].set_title(filenames[index])
plt.show()

In [ ]:
disps = [cv2_imshow_mask(images[i], outputs[i], display=False) for i in indices]

In [ ]:
# display samples
fig, axs = plt.subplots(1, len(disps), figsize=(20, 10))
for i, disp in enumerate(disps):
    axs[i].imshow(cv2.cvtColor(disp,cv2.COLOR_BGR2RGB))
    axs[i].axis('off')
    axs[i].set_title(filenames[indices[i]])
plt.show()

In [ ]:
# display target masks
fig, axs = plt.subplots(1, len(indices), figsize=(20, 10))
for i, index in enumerate(indices):
    axs[i].imshow(outputs[index], cmap='gray')
    axs[i].axis('off')
    axs[i].set_title(filenames[index])
plt.show()

In [ ]:
class ListDataset(Dataset):
    def __init__(self, inputs, targets, transform=None):
        self.inputs = inputs
        self.targets = targets
        self.transform = transform

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        x = self.inputs[idx]
        y = self.targets[idx]

        if self.transform:
            x = self.transform(x)

        return x, y

In [ ]:
dataset = ListDataset(inputs, outputs)

In [ ]:
loader = DataLoader(dataset, batch_size=1, shuffle=True) # different size of images

In [ ]:
# architecture

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(), #nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(), #nn.GELU(),
        )

    def forward(self, x):
        return self.block(x)


class UpsampleBlock(nn.Module):
    def __init__(self, in_ch, out_ch, scale):
        super().__init__()
        self.scale = scale
        self.conv = ConvBlock(in_ch, out_ch)

    def forward(self, x):
        x = F.interpolate(x, scale_factor=self.scale, mode="bilinear", align_corners=False)
        x = self.conv(x)
        return x


class FeatureExpander(nn.Module):
    def __init__(self):
        super().__init__()

        self.up1 = UpsampleBlock(384, 256, scale=2)   # 128 → 256
        self.up2 = UpsampleBlock(256, 128, scale=3)   # 256 → 768
        self.out = nn.Conv2d(128, 32, kernel_size=1) #16

    def forward(self, x):
        # input: B x 384 x 128 x 128
        x = self.up1(x)   # B x 256 x 256 x 256
        x = self.up2(x)   # B x 128 x 768 x 768
        x = self.out(x)   # B x 16 x 768 x 768
        return x

In [ ]:
expander = FeatureExpander().to(device)

In [ ]:
# loss function
criterion = nn.MSELoss()

In [ ]:
# optimizer
optimizer = optim.Adam(expander.parameters(), lr=0.001)

In [ ]:
# scheduler
step_size = 40
gamma = 0.1
scheduler = StepLR(optimizer, step_size=step_size, gamma=gamma)

In [ ]:
expander.train()

In [ ]:
x_train, y_train = next(iter(loader))

In [ ]:
x_train.shape, y_train.shape

In [ ]:
input = x_train.to(device)

In [ ]:
optimizer.zero_grad()

In [ ]:
output = expander(input)

In [ ]:
output.shape

In [ ]:
feats = F.interpolate(output, size=(y_train.shape[1], y_train.shape[2]), mode="bilinear", align_corners=False)

In [ ]:
feats.shape

In [ ]:
result = ncut(feats, data_format='bchw', num_iters=1, auto_fix=False, return_all=True)

In [ ]:
b = result['b']
eigenvector = result['eigenvector']
d = result['d']

In [ ]:
b.shape, b[0].item()

In [ ]:
eigenvector.shape, eigenvector.max(), eigenvector.min()

In [ ]:
d.shape, d.max(), d.min()

In [ ]:
expected_output = y_train.to(device)

In [ ]:
expected_output.shape

In [ ]:
expected_eigenvector, b = targetFromMask(expected_output,d)

In [ ]:
b.shape, b[0].item()

In [ ]:
expected_eigenvector.shape, expected_eigenvector.max().item(), expected_eigenvector.min().item()

In [ ]:
# training
epoch = 0
history = []
auto_b = False
auto_fix = False

def train(num_epochs):
    global epoch
    for _ in range(num_epochs):
        # change model in training mood
        expander.train()

        # to record loss and accuracy
        batch_loss = []
        for x_train, y_train in loader:

            # send data to device
            input = x_train.to(device)

            # reset parameters gradient to zero
            optimizer.zero_grad()

            # forward pass to the model
            output = expander(input)
            feats = F.interpolate(output, size=(y_train.shape[1], y_train.shape[2]), mode="bilinear", align_corners=False)
            result = ncut(feats, data_format='bchw', num_iters=1, auto_fix=auto_fix, return_all=True)
            eigenvector = result['eigenvector']

            expected_output = y_train.to(device)
            if auto_b:
                b = result['b']
                expected_eigenvector, _ = targetFromMask(expected_output, custom_b=b)
            else:
                d = result['d']
                expected_eigenvector, _ = targetFromMask(expected_output, d=d)

            # mse loss
            loss = criterion(eigenvector, expected_eigenvector)

            # find gradients
            loss.backward()
            # update parameters using gradients
            optimizer.step()

            # recording loss
            batch_loss.append(loss.item())

        epoch_loss = np.average(batch_loss)
        history.append(epoch_loss)

        expander.eval()

        print(f'Epoch: {epoch} Loss: {1000000*epoch_loss:.6f} Learning Rate: {scheduler.get_last_lr()[0]:.7f}')
        scheduler.step()
        epoch += 1

In [ ]:
train(120)

In [ ]:
# plot history
plt.plot(history)
plt.title('matting model loss')
plt.ylabel('Loss (MSE x 10^6)')
plt.xlabel('Epoch')

In [ ]:
torch.save(expander,'expander.pth')

In [ ]:
backbone.to(device)

In [ ]:
# try model
def matting(image):
    with torch.no_grad():
        blob = transform(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)).unsqueeze(0).to(device)
        feats = backbone.get_intermediate_layers(blob,norm=False)[0]
        feats = feats.view(feats.shape[0],blob.shape[2]//backbone.patch_size,blob.shape[3]//backbone.patch_size,feats.shape[-1])
        feats = feats.permute(0,3,1,2)
        output = expander(feats)
        feats2 = F.interpolate(output, size=(image.shape[0], image.shape[1]), mode="bilinear", align_corners=False)
        result = ncut(feats2, data_format='bchw', num_iters=1, auto_fix=auto_fix)
        bipartition = result[0].cpu().numpy().astype(np.uint8)*255
    return bipartition

In [ ]:
masks = [matting(images[i]) for i in indices]

In [ ]:
disps = [cv2_imshow_mask(images[index], masks[i], display=False) for i, index in enumerate(indices)]

In [ ]:
# display results
fig, axs = plt.subplots(1, len(disps), figsize=(20, 10))
for i, disp in enumerate(disps):
    axs[i].imshow(cv2.cvtColor(disp,cv2.COLOR_BGR2RGB))
    axs[i].axis('off')
    axs[i].set_title(filenames[indices[i]])
plt.show()

In [ ]:
# display calculated masks
fig, axs = plt.subplots(1, len(indices), figsize=(20, 10))
for i, index in enumerate(indices):
    axs[i].imshow(masks[i], cmap='gray')
    axs[i].axis('off')
    axs[i].set_title(filenames[index])
plt.show()

In [ ]:
!wget -O input.jpg https://github.com/andylucny/BackgroundRemovalByModNet4OpenCV/blob/main/input.jpg?raw=true

In [ ]:
input = cv2.imread('input.jpg')

In [ ]:
mask = matting(input)

In [ ]:
cv2_imshow_mask(input, mask)

In [ ]:
cv2_imshow(mask)

In [ ]:
from google.colab import files
files.download('expander.pth')

In [ ]:
class CombinedModel(nn.Module):
    def __init__(self, backbone, expander):
        super().__init__()
        self.mean = [0.485, 0.456, 0.406]
        self.std = [0.229, 0.224, 0.225]
        self.backbone = backbone
        self.size = (768, 768)
        self.patch_size = backbone.patch_size
        self.expander = expander
        self.ncut = Ncut(num_iters=1)

    def forward(self, x):
        B, C, H, W = x.shape
        x = x.float()/255
        x = F.interpolate(x, size=self.size, mode='bilinear', align_corners=False)
        x[:,0] = (x[:,0] - self.mean[0]) / self.std[0]
        x[:,1] = (x[:,1] - self.mean[1]) / self.std[1]
        x[:,2] = (x[:,2] - self.mean[2]) / self.std[2]
        x = self.backbone.get_intermediate_layers(x,norm=False)[0]
        x = x.reshape(B,self.size[0]//self.patch_size,self.size[1]//self.patch_size,-1).permute(0,3,1,2)
        x = self.expander(x)
        x = F.interpolate(x, size=(H, W), mode="bilinear", align_corners=False)
        x = self.ncut(x)
        return x

In [ ]:
model = CombinedModel(backbone,expander).to(device)
model.eval()

In [ ]:
blob = torch.tensor(cv2.cvtColor(input, cv2.COLOR_BGR2RGB)).float().permute(2,0,1).unsqueeze(0).to(device)
blob.shape

In [ ]:
bipartition = model(blob)

In [ ]:
mask = bipartition[0].cpu().numpy().astype(np.uint8)*255

In [ ]:
mask.shape, input.shape

In [ ]:
plt.imshow(cv2.cvtColor(cv2_imshow_mask(input, mask, display=False),cv2.COLOR_BGR2RGB))

In [ ]:
torch.save(model,"fgsegmentation.pth")

In [ ]:
from google.colab import files
files.download('fgsegmentation.pth')